# 📈 Proyecto de Forecasting: Pipeline de Inferencia para Producción

Este notebook contiene el flujo de transformación y preparación de datos para los nuevos registros del año 2025. El objetivo principal es replicar con exactitud la ingeniería de variables desarrollada en el módulo de entrenamiento, asegurando que la matriz de características mantenga una consistencia estructural absoluta con el modelo predictivo final antes de ser exportada.

In [1]:
# Importación de librerías necesarias y definición de rutas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
import streamlit as st
import holidays
from pathlib import Path

ruta_datos = Path('../data/raw/inferencia')
ruta_procesados = Path('../data/processed')

## Carga de datos y preparación del dataset

Importamos los registros de ventas correspondientes al periodo de inferencia 2025. En esta primera fase, realizamos la conversión de tipos a formato cronológico y ordenamos el dataset por producto y tiempo. Esta disposición secuencial es obligatoria para garantizar la coherencia matemática en el cálculo posterior de las variables dependientes del tiempo.

In [2]:
# Carga del dataset de inferencia y conversión a formato cronológico
inferencia_df = pd.read_csv(ruta_datos / 'ventas_2025_inferencia.csv')

inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df = inferencia_df.sort_values(['producto_id', 'fecha']).reset_index(drop=True)

display(inferencia_df.head())

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage,año
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78,2025
1,2025-10-26,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,20.0,105.75,2115.00,86.44,114.57,94.06,2025
2,2025-10-27,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,16.0,114.95,1839.20,86.91,112.53,93.78,2025
3,2025-10-28,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,15.0,117.31,1759.65,82.31,118.68,108.45,2025
4,2025-10-29,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,14.0,108.10,1513.40,83.96,111.43,106.17,2025


## Creación de Variables Nuevas (Feature Engineering) - Componentes de Calendario

Construimos las variables temporales básicas para aislar el comportamiento de la demanda según el momento del tiempo. Utilizando librerías de gestión de calendarios, identificamos de forma automática los días festivos oficiales en España y calculamos la posición exacta de hitos críticos de alta volatilidad comercial en el e-commerce, como el Black Friday y el Cyber Monday.

In [3]:
# Construcción de variables temporales y cálculo de festivos y campañas
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_del_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['es_fin_semana'] = inferencia_df['fecha'].dt.dayofweek.isin([5, 6])
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['semana_del_año'] = inferencia_df['fecha'].dt.isocalendar().week.astype(int)
inferencia_df['es_inicio_mes'] = inferencia_df['fecha'].dt.is_month_start
inferencia_df['es_fin_mes'] = inferencia_df['fecha'].dt.is_month_end

años = range(inferencia_df['año'].min(), inferencia_df['año'].max() + 1)
calendario_espana = holidays.country_holidays('ES', years=años)
inferencia_df['es_festivo'] = inferencia_df['fecha'].dt.date.isin(set(calendario_espana.keys()))

def obtener_black_friday(year):
    primer_dia = pd.Timestamp(year=year, month=11, day=1)
    primer_jueves = primer_dia + pd.to_timedelta((3 - primer_dia.weekday()) % 7, unit='D')
    thanksgiving = primer_jueves + pd.to_timedelta(21, unit='D')
    return thanksgiving + pd.to_timedelta(1, unit='D')

inferencia_df['es_Black_Friday'] = inferencia_df['fecha'].isin([obtener_black_friday(year) for year in años])
inferencia_df['es_Cyber_Monday'] = inferencia_df['fecha'].isin([obtener_black_friday(year) + pd.to_timedelta(3, unit='D') for year in años])
inferencia_df['es_laborable'] = ~inferencia_df['es_fin_semana'] & ~inferencia_df['es_festivo']

## Creación de Variables Nuevas (Feature Engineering) - Inercia y Precios

Calculamos las variables basadas en el histórico de ventas realizando los cálculos de forma independiente para cada producto. Este bloque computa la media móvil de los últimos 7 días y las ventanas de desfase (lags), lo que permite al modelo entender el ritmo y la tendencia de ventas reciente de cada artículo por separado. Asimismo, estructuramos una métrica para comparar nuestro precio de venta frente al promedio de los competidores.

In [4]:
# Cálculo de históricos de ventas (lags/medias) y relaciones de precios con la competencia
inferencia_df['unidades_vendidas_ma7'] = (
    inferencia_df.groupby('producto_id')['unidades_vendidas']
    .transform(lambda s: s.shift(1).rolling(window=7, min_periods=1).mean())
)

for lag in range(1, 8):
    inferencia_df[f'unidades_vendidas_lag{lag}'] = inferencia_df.groupby('producto_id')['unidades_vendidas'].shift(lag)

inferencia_df['descuento_pct'] = ((inferencia_df['precio_base'] - inferencia_df['precio_venta']) / inferencia_df['precio_base'].replace({0: np.nan}))
inferencia_df['descuento_pct'] = inferencia_df['descuento_pct'].replace([np.inf, -np.inf], np.nan).fillna(0)
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']


## Codificación de Variables Categóricas (One-Hot Encoding)

Restringimos el conjunto de datos al mes de noviembre, alineándonos con el foco estratégico y el escenario de validación del modelo de predicción de campañas. Posteriormente, aplicamos transformaciones de One-Hot Encoding sobre las variables cualitativas del catálogo para convertirlas en los vectores binarios numéricos exigidos por el algoritmo.

In [5]:
# Aislamiento de la campaña de noviembre y transformación a variables dummies
inferencia_df = inferencia_df.loc[inferencia_df['mes'] == 11].copy()

for columna in ['nombre', 'categoria', 'subcategoria']:
    inferencia_df[f'{columna}_h'] = inferencia_df[columna]

inferencia_df = pd.get_dummies(
    inferencia_df,
    columns=['nombre_h', 'categoria_h', 'subcategoria_h'],
    prefix=['nombre_h', 'categoria_h', 'subcategoria_h'],
    prefix_sep='='
)

## Preparación de la matriz de características y exportación

En este último paso, consolidamos el dataset de inferencia final manteniendo la estructura completa de variables generadas. Posteriormente, exportamos la matriz procesada en formato CSV hacia la ruta de almacenamiento, quedando disponible para ser cargada e interpretada por la aplicación interactiva de Streamlit.

In [6]:
# Consolidación de la matriz final de características
X_inferencia = inferencia_df.copy()

# Almacenamiento del dataset procesado de inferencia
ruta_procesados.mkdir(parents=True, exist_ok=True)
X_inferencia.to_csv(ruta_procesados / 'inferencia_df_transformado.csv', index=False)

# Validación de las dimensiones finales de la matriz de datos
print("Dimensiones de la matriz de inferencia:")
print(X_inferencia.shape)
display(X_inferencia.head())

Dimensiones de la matriz de inferencia:
(720, 81)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h=Esterilla Yoga,subcategoria_h=Mancuernas Ajustables,subcategoria_h=Mochila Trekking,subcategoria_h=Pesa Rusa,subcategoria_h=Pesas Casa,subcategoria_h=Rodillera Yoga,subcategoria_h=Ropa Montaña,subcategoria_h=Ropa Running,subcategoria_h=Zapatillas Running,subcategoria_h=Zapatillas Trail
7,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
8,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
9,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
10,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
11,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
